In [17]:
import json
import numpy as np

EMBED_FILE = "../data/embeddings/ministry_embeddings3.json"

def load_embeddings():
    with open(EMBED_FILE, "r") as f:
        data = json.load(f)

    return data

data = load_embeddings()

all_vectors = []
vector_ministry = []

for ministry, vectors in data.items():
    for v in vectors:
        all_vectors.append(v)
        vector_ministry.append(ministry)

X = np.array(all_vectors)

print("Total paragraph vectors:", X.shape)

Total paragraph vectors: (1741, 384)


In [18]:
print("Total NaNs in X:", np.isnan(X).sum())

Total NaNs in X: 0


In [19]:
nan_rows = np.isnan(X).any(axis=1)

print("Number of bad vectors:", nan_rows.sum())

Number of bad vectors: 0


In [21]:
zero_rows = np.where(np.linalg.norm(X, axis=1) == 0)[0]
print("Zero vectors:", len(zero_rows))

Zero vectors: 0


In [8]:
from collections import defaultdict

In [9]:
def print_clusters(names, labels):
    clusters = {}
    for name, label in zip(names, labels):
        clusters.setdefault(label, []).append(name)

    for k, v in clusters.items():
        print(f"\nCluster {k}:")
        for m in v:
            print("  ", m)

In [10]:
from sklearn.cluster import KMeans

def kmeans_clustering(X, k=8):
    model = KMeans(n_clusters=k, random_state=42)
    labels = model.fit_predict(X)

    ministry_cluster_count = defaultdict(lambda: defaultdict(int))

    for ministry, cluster in zip(vector_ministry, labels):
        ministry_cluster_count[ministry][cluster] += 1
        
    ministry_vectors = {}
    num_clusters = k

    for ministry in data.keys():
        vec = np.zeros(num_clusters)
        total = sum(ministry_cluster_count[ministry].values())
        
        for c, count in ministry_cluster_count[ministry].items():
            vec[c] = count / total
        
        ministry_vectors[ministry] = vec
        
    names = list(ministry_vectors.keys())
    X_min = np.vstack([ministry_vectors[m] for m in names])

    labels_ministry = KMeans(n_clusters=k, random_state=42).fit_predict(X_min)
    return names, labels_ministry

names, labels_kmeans = kmeans_clustering(X, k=10)
print_clusters(names, labels_kmeans)


Cluster 3:
   Ministry_of_Power_MoP_
   Ministry_of_Heavy_Industries_MoHI_
   Ministry_of_Mines_MoM_
   Ministry_of_Petroleum_and_Natural_Gas_MoPNG_
   Ministry_of_Steel_MoS_
   Ministry_of_Railways_MoR_
   Ministry_of_Coal_MoC_
   Ministry_of_Ports_Shipping_and_Waterways_MoPSW_
   Ministry_of_New_and_Renewable_Energy_MNRE_

Cluster 0:
   Ministry_of_Micro_Small_Medium_Enterprises_MoME_
   Ministry_of_Electronics_and_Information_Technology_MeitY_
   Ministry_of_Minority_Affairs_MoMA_
   Ministry_of_Development_of_North_Eastern_Region_MDoNER_
   Ministry_of_Communications_MoC_
   Ministry_of_Defence_MoD_
   Ministry_of_Civil_Aviation_MoCA_

Cluster 5:
   Ministry_of_AYUSH_MoA_
   Ministry_of_Social_Justice_and_Empowerment_MoSJE_
   Ministry_of_Women_and_Child_Development_MoWCD_
   Ministry_of_Health_and_Family_Welfare_MoHFW_
   Ministry_of_Tribal_Affairs_MoTA_

Cluster 2:
   Ministry_of_Textiles_MoT_
   Ministry_of_Food_Processing_Industries_MoFPI_
   Ministry_of_Consumer_Affairs_Food_

In [11]:
from sklearn.mixture import GaussianMixture

def gmm_clustering(X, k=10):
    model = GaussianMixture(n_components=k, covariance_type='full', random_state=42)
    labels = model.fit_predict(X)

    ministry_cluster_count = defaultdict(lambda: defaultdict(int))

    for ministry, cluster in zip(vector_ministry, labels):
        ministry_cluster_count[ministry][cluster] += 1
        
    ministry_vectors = {}
    num_clusters = k

    for ministry in data.keys():
        vec = np.zeros(num_clusters)
        total = sum(ministry_cluster_count[ministry].values())
        
        for c, count in ministry_cluster_count[ministry].items():
            vec[c] = count / total
        
        ministry_vectors[ministry] = vec
        
    names = list(ministry_vectors.keys())
    X_min = np.vstack([ministry_vectors[m] for m in names])

    labels_ministry = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit_predict(X_min)
    return names, labels_ministry

names, labels_gmm = gmm_clustering(X, k=10)
print_clusters(names, labels_gmm)


Cluster 3:
   Ministry_of_Power_MoP_
   Ministry_of_Heavy_Industries_MoHI_
   Ministry_of_Mines_MoM_
   Ministry_of_Petroleum_and_Natural_Gas_MoPNG_
   Ministry_of_Steel_MoS_
   Ministry_of_Railways_MoR_
   Ministry_of_Coal_MoC_
   Ministry_of_Ports_Shipping_and_Waterways_MoPSW_
   Ministry_of_New_and_Renewable_Energy_MNRE_

Cluster 0:
   Ministry_of_Micro_Small_Medium_Enterprises_MoME_
   Ministry_of_Electronics_and_Information_Technology_MeitY_
   Ministry_of_Minority_Affairs_MoMA_
   Ministry_of_Development_of_North_Eastern_Region_MDoNER_
   Ministry_of_Communications_MoC_
   Ministry_of_Defence_MoD_
   Ministry_of_Civil_Aviation_MoCA_

Cluster 5:
   Ministry_of_AYUSH_MoA_
   Ministry_of_Social_Justice_and_Empowerment_MoSJE_
   Ministry_of_Women_and_Child_Development_MoWCD_
   Ministry_of_Health_and_Family_Welfare_MoHFW_
   Ministry_of_Tribal_Affairs_MoTA_

Cluster 2:
   Ministry_of_Textiles_MoT_
   Ministry_of_Food_Processing_Industries_MoFPI_
   Ministry_of_Consumer_Affairs_Food_

In [12]:
from sklearn.cluster import AgglomerativeClustering

def hierarchical_clustering(X, k=10):
    model = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = model.fit_predict(X)

    ministry_cluster_count = defaultdict(lambda: defaultdict(int))

    for ministry, cluster in zip(vector_ministry, labels):
        ministry_cluster_count[ministry][cluster] += 1
        
    ministry_vectors = {}
    num_clusters = k

    for ministry in data.keys():
        vec = np.zeros(num_clusters)
        total = sum(ministry_cluster_count[ministry].values())
        
        for c, count in ministry_cluster_count[ministry].items():
            vec[c] = count / total
        
        ministry_vectors[ministry] = vec
        
    names = list(ministry_vectors.keys())
    X_min = np.vstack([ministry_vectors[m] for m in names])

    labels_ministry = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X_min)
    return names, labels_ministry

names, labels_hier = hierarchical_clustering(X, k=10)
print_clusters(names, labels_hier)


Cluster 1:
   Ministry_of_Power_MoP_
   Ministry_of_Heavy_Industries_MoHI_
   Ministry_of_Mines_MoM_
   Ministry_of_Petroleum_and_Natural_Gas_MoPNG_
   Ministry_of_Steel_MoS_
   Ministry_of_Coal_MoC_
   Ministry_of_New_and_Renewable_Energy_MNRE_

Cluster 2:
   Ministry_of_Micro_Small_Medium_Enterprises_MoME_
   Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_
   Ministry_of_Law_and_Justice_MoLJ_
   Ministry_of_Home_Affairs_MHA_
   Ministry_of_Cooperation_MoC_
   Ministry_of_Information_and_Broadcasting_MIB_
   Ministry_of_Finance_MoF_
   Ministry_of_Parliamentary_Affairs_MPA_
   Ministry_of_Corporate_Affairs_MCA_
   Ministry_of_Labour_and_Employment_MoLE_

Cluster 6:
   Ministry_of_AYUSH_MoA_
   Ministry_of_Health_and_Family_Welfare_MoHFW_

Cluster 0:
   Ministry_of_Textiles_MoT_
   Ministry_of_Food_Processing_Industries_MoFPI_
   Ministry_of_Consumer_Affairs_Food_and_Public_Distribution_MoCAF_PD_
   Ministry_of_Chemicals_and_Fertilizers_MoCF_
   Ministry_of_Agriculture_an

In [13]:
from sklearn.cluster import DBSCAN

def dbscan_clustering(X, eps=0.5, min_samples=3):
    model = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine')
    labels = model.fit_predict(X)

    ministry_cluster_count = defaultdict(lambda: defaultdict(int))

    for ministry, cluster in zip(vector_ministry, labels):
        ministry_cluster_count[ministry][cluster] += 1
        
    ministry_vectors = {}
    num_clusters = k

    for ministry in data.keys():
        vec = np.zeros(num_clusters)
        total = sum(ministry_cluster_count[ministry].values())
        
        for c, count in ministry_cluster_count[ministry].items():
            vec[c] = count / total
        
        ministry_vectors[ministry] = vec
        
    names = list(ministry_vectors.keys())
    X_min = np.vstack([ministry_vectors[m] for m in names])

    labels_ministry = DBSCAN(eps=eps, min_samples=min_samples, metric='cosine').fit_predict(X_min)
    return names, labels_ministry

names, labels_dbscan = dbscan_clustering(X)
print_clusters(names, labels_dbscan)


Cluster 0:
   Ministry_of_Power_MoP_
   Ministry_of_Micro_Small_Medium_Enterprises_MoME_
   Ministry_of_AYUSH_MoA_
   Ministry_of_Textiles_MoT_
   Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_
   Ministry_of_Law_and_Justice_MoLJ_
   Ministry_of_Social_Justice_and_Empowerment_MoSJE_
   Ministry_of_Culture
   Ministry_of_Heavy_Industries_MoHI_
   Ministry_of_Statistics_and_Programme_Implementation_MoSPI_
   Ministry_of_Tourism_MoT_
   Ministry_of_External_Affairs_MEA_
   Ministry_of_Food_Processing_Industries_MoFPI_
   Ministry_of_Skill_Development_and_Entrepreneurship_MSDE_
   Ministry_of_Mines_MoM_
   Ministry_of_Consumer_Affairs_Food_and_Public_Distribution_MoCAF_PD_
   Ministry_of_Women_and_Child_Development_MoWCD_
   Ministry_of_Home_Affairs_MHA_
   Ministry_of_Petroleum_and_Natural_Gas_MoPNG_
   Ministry_of_Electronics_and_Information_Technology_MeitY_
   Ministry_of_Youth_Affairs_and_Sports_MoYAS_
   Ministry_of_Minority_Affairs_MoMA_
   Ministry_of_Rural_Developme

In [24]:
all_labels = {
    "kmeans": labels_kmeans,
    "gmm": labels_gmm,
    "hier": labels_hier
}

In [25]:
n = len(names)
co_matrix = np.zeros((n, n))

for algo, labels in all_labels.items():
    for i in range(n):
        for j in range(n):
            if labels[i] == labels[j]:
                co_matrix[i][j] += 1

In [26]:
co_matrix = co_matrix / len(all_labels)

In [27]:
threshold = 0.8  # adjust

pairs = []
for i in range(n):
    for j in range(i+1, n):
        if co_matrix[i][j] >= threshold:
            pairs.append((names[i], names[j]))

print("Strong pairs:")
for p in pairs:
    print(p)

Strong pairs:
('Ministry_of_Power_MoP_', 'Ministry_of_Heavy_Industries_MoHI_')
('Ministry_of_Power_MoP_', 'Ministry_of_Mines_MoM_')
('Ministry_of_Power_MoP_', 'Ministry_of_Petroleum_and_Natural_Gas_MoPNG_')
('Ministry_of_Power_MoP_', 'Ministry_of_Steel_MoS_')
('Ministry_of_Power_MoP_', 'Ministry_of_Coal_MoC_')
('Ministry_of_Power_MoP_', 'Ministry_of_New_and_Renewable_Energy_MNRE_')
('Ministry_of_AYUSH_MoA_', 'Ministry_of_Health_and_Family_Welfare_MoHFW_')
('Ministry_of_Textiles_MoT_', 'Ministry_of_Food_Processing_Industries_MoFPI_')
('Ministry_of_Textiles_MoT_', 'Ministry_of_Consumer_Affairs_Food_and_Public_Distribution_MoCAF_PD_')
('Ministry_of_Textiles_MoT_', 'Ministry_of_Chemicals_and_Fertilizers_MoCF_')
('Ministry_of_Textiles_MoT_', 'Ministry_of_Agriculture_and_Farmers_Welfare_MoAFW_')
('Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_', 'Ministry_of_Home_Affairs_MHA_')
('Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_', 'Ministry_of_Cooperation_MoC_')
('Mi

In [ ]:
# Due to high semantic overlap in ministry embeddings, traditional clustering methods produced weak separation (silhouette < 0.1).
# Therefore, we adopted a consensus-based clustering approach to identify stable ministry groupings.

In [28]:
import networkx as nx

G = nx.Graph()

for i in range(n):
    G.add_node(names[i])

for i in range(n):
    for j in range(i+1, n):
        if co_matrix[i][j] >= 0.8:
            G.add_edge(names[i], names[j])

components = list(nx.connected_components(G))

for i, comp in enumerate(components):
    print(f"\nStable Cluster {i}:")
    for m in comp:
        print(" ", m)


Stable Cluster 0:
  Ministry_of_Petroleum_and_Natural_Gas_MoPNG_
  Ministry_of_Steel_MoS_
  Ministry_of_Mines_MoM_
  Ministry_of_Power_MoP_
  Ministry_of_New_and_Renewable_Energy_MNRE_
  Ministry_of_Coal_MoC_
  Ministry_of_Heavy_Industries_MoHI_

Stable Cluster 1:
  Ministry_of_Micro_Small_Medium_Enterprises_MoME_

Stable Cluster 2:
  Ministry_of_AYUSH_MoA_
  Ministry_of_Health_and_Family_Welfare_MoHFW_

Stable Cluster 3:
  Ministry_of_Chemicals_and_Fertilizers_MoCF_
  Ministry_of_Agriculture_and_Farmers_Welfare_MoAFW_
  Ministry_of_Textiles_MoT_
  Ministry_of_Consumer_Affairs_Food_and_Public_Distribution_MoCAF_PD_
  Ministry_of_Food_Processing_Industries_MoFPI_

Stable Cluster 4:
  Ministry_of_Personnel_Public_Grievances_and_Pensions_MoPPGP_
  Ministry_of_Labour_and_Employment_MoLE_
  Ministry_of_Information_and_Broadcasting_MIB_
  Ministry_of_Cooperation_MoC_
  Ministry_of_Home_Affairs_MHA_

Stable Cluster 5:
  Ministry_of_Parliamentary_Affairs_MPA_
  Ministry_of_Finance_MoF_
  Mini